This notebook uses the same raw rows that produced `llama_tabular_outputs.csv`, but prompts the fine-tuned Llama LoRA adapter with a Markdown table instead of the original plain text `input_text` prompt.

### Run this only if the runtime does not already have the project dependencies. In Colab, make sure the runtime type is GPU.



In [ ]:
# Optional: uncomment if needed
# %pip install -r ../../requirements.txt
# %pip install -U bitsandbytes>=0.46.1

### Configure Paths, Token, And Adapter

`ADAPTER_DIR` must point to the trained LoRA adapter folder. The default matches the training script output.

In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
elif PROJECT_ROOT.name == "llama":
    PROJECT_ROOT = PROJECT_ROOT.parent

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
INPUT_CSV = PROJECT_ROOT / "data" / "mimic_labs_20_for_testing.csv"
ADAPTER_DIR = PROJECT_ROOT / "llama" / "outputs" / "llama-lab-lora"
OUTPUT_CSV = PROJECT_ROOT / "llama" / "outputs" / "tabular_prompt_approach" / "llama_tabular_finetuned_outputs.csv"
MAX_ROWS = 10
MAX_NEW_TOKENS = 200
USE_4BIT = True

# If your runtime does not inherit HF_TOKEN, temporarily set it here.
# os.environ["HF_TOKEN"] = "hf_your_token_here"

print("Project root:", PROJECT_ROOT)
print("Input:", INPUT_CSV)
print("Adapter:", ADAPTER_DIR)
print("Output:", OUTPUT_CSV)
print("HF_TOKEN set:", bool(os.getenv("HF_TOKEN")))
print("Adapter exists:", ADAPTER_DIR.exists())

### Load The Same Raw Input Rows As For Text-only Prompt


In [ ]:
import pandas as pd

df = pd.read_csv(INPUT_CSV).head(MAX_ROWS).copy()
print("Rows loaded:", len(df))
df.head(3)

### Build The Table-Style Prompt

In [ ]:
SYSTEM_PROMPT = (
    "You produce concise, patient-friendly explanations of laboratory results. "
    "Use cautious wording, avoid diagnosis and do not recommend treatment."
)

def has_value(value):
    return not pd.isna(value) and str(value).strip() and str(value).strip().lower() != "nan"

def value_from(row, *columns, default=""):
    for column in columns:
        if column in row and has_value(row[column]):
            return str(row[column]).strip()
    return default

def markdown_cell(value):
    text = str(value).replace("\r", " ").replace("\n", " ").strip()
    return text.replace("|", "\\|")

def build_tabular_prompt(row):
    measured_value = " ".join(
        part
        for part in (
            value_from(row, "VALUE", "value", default="unknown"),
            value_from(row, "VALUEUOM", "unit", "units", default=""),
        )
        if part
    )
    table = [
        "| field | value |",
        "| --- | --- |",
        f"| Patient sex | {markdown_cell(value_from(row, 'GENDER', 'gender', 'sex'))} |",
        f"| Admission type | {markdown_cell(value_from(row, 'ADMISSION_TYPE', 'admission_type'))} |",
        f"| Admission diagnosis | {markdown_cell(value_from(row, 'DIAGNOSIS', 'diagnosis'))} |",
        f"| Laboratory test | {markdown_cell(value_from(row, 'lab_name', 'LAB_NAME', 'test_name', 'label'))} |",
        f"| Fluid | {markdown_cell(value_from(row, 'fluid', 'FLUID'))} |",
        f"| Category | {markdown_cell(value_from(row, 'category', 'CATEGORY'))} |",
        f"| Measured value | {markdown_cell(measured_value)} |",
        f"| Abnormal flag | {markdown_cell(value_from(row, 'FLAG', 'flag', default='not available'))} |",
    ]
    return (
        "Read the laboratory result in the table below.\n\n"
        + "\n".join(table)
        + "\n\nTask: Write one short, patient-friendly explanation of what this result may mean for the body.\n"
        "Rules:\n"
        "- Use simple language.\n"
        "- Use cautious wording such as may suggest, can suggest, may reflect, or appears.\n"
        "- Do not diagnose disease.\n"
        "- Do not recommend treatment.\n"
        "- Keep the answer to one or two short sentences."
    )

In [ ]:
example_prompt = build_tabular_prompt(df.iloc[0])
print(example_prompt)

### Load Base Llama And Fine-Tuned LoRA Adapter


In [ ]:
import torch
import transformers.utils.import_utils as transformers_import_utils
import transformers.utils as transformers_utils
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

transformers_import_utils.is_torchvision_available = lambda: False
transformers_utils.is_torchvision_available = lambda: False

token = (os.getenv("HF_TOKEN") or "").strip()
if not token:
    raise EnvironmentError("Set HF_TOKEN before loading Llama weights from Hugging Face.")
if not ADAPTER_DIR.exists():
    raise FileNotFoundError(f"LoRA adapter folder not found: {ADAPTER_DIR}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

supports_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if supports_bf16 else torch.float16
quantization_config = None

if USE_4BIT:
    if not torch.cuda.is_available():
        raise RuntimeError("4-bit adapter generation requires a CUDA GPU. Set USE_4BIT = False for CPU/full precision.")
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype,
    )

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=token,
    dtype=compute_dtype if torch.cuda.is_available() else torch.float32,
    quantization_config=quantization_config,
    device_map={"": 0} if USE_4BIT else "auto",
)
base_model.eval()

tuned_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
tuned_model.eval()
print("Loaded base model and LoRA adapter")

### Generate One Example

In [ ]:
def build_chat_prompt(user_prompt):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def generate(model, prompt, max_new_tokens=MAX_NEW_TOKENS):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0, inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

chat_prompt = build_chat_prompt(example_prompt)
one_output = generate(tuned_model, chat_prompt)
print(one_output)

### Generate And Save All Rows


In [ ]:
df["fine_tuned_llama_tabular_prompt"] = ""
df["fine_tuned_llama_tabular_output"] = ""

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

for row_index in range(len(df)):
    tabular_prompt = build_tabular_prompt(df.iloc[row_index])
    model_prompt = build_chat_prompt(tabular_prompt)
    df.at[row_index, "fine_tuned_llama_tabular_prompt"] = model_prompt
    df.at[row_index, "fine_tuned_llama_tabular_output"] = generate(tuned_model, model_prompt)
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"Processed row {row_index + 1}/{len(df)}")

print("Wrote:", OUTPUT_CSV)
df[["lab_name", "VALUE", "VALUEUOM", "FLAG", "fine_tuned_llama_tabular_output"]].head()